---
title: "Économétrie Financière"
author: "Corrigé — Fiche TP #1 (Python)"
subtitle: "Données financières et faits stylisés sur données réelles"
format:
  html:
    toc: true
    toc-depth: 2
    code-fold: false
    embed-resources: true
jupyter: python3
execute:
  warning: false
  message: false
---

> Pour obtenir un notebook Jupyter à partir de ce fichier :
> `quarto convert TP1-donnees-faits-stylises-python-corrige.qmd`

In [ ]:
#| label: install library python
# a ne lancer qu'une fois, que si les librairies ne sont pas installées
# enlever le # devant %pip install pour installer les librairies
# %pip install numpy pandas matplotlib scipy statsmodels openpyxl
# 

In [1]:
#| label: packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from statsmodels.stats.stattools import jarque_bera
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams["figure.figsize"] = (10, 6)

# Exercice 1 — Prix, rendements simples et log-rendements

## (a) Import des prix

In [2]:
#| label: donnees
FCHI = yf.download("^FCHI", start="2018-01-01", end="2023-12-31", auto_adjust=True)
cac_prices = FCHI["Close"].squeeze()  # DataFrame à une colonne -> Series, quelle
                                       # que soit la version de yfinance installée

[*********************100%***********************]  1 of 1 completed


## (b) Rendements simples et log-rendements

In [ ]:
#| label: rendements
cac_simple = cac_prices.pct_change().dropna() * 100
cac_log = np.log(cac_prices).diff().dropna() * 100

fig, ax = plt.subplots()
ax.plot(cac_log, color="darkred", lw=1, label="Log-rendement")
ax.plot(cac_simple, color="steelblue", lw=1, ls="--", label="Rendement simple")
ax.set_title("CAC40 — rendement simple vs log-rendement (%)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

Les deux courbes sont quasiment superposées — visuellement
indissociables l'une de l'autre en dehors des jours de très forte
variation.

## (c) Écart maximal entre les deux définitions

In [ ]:
#| label: ecart-max
ecart = (cac_simple - cac_log).abs()
ecart = ecart.dropna()
i_max = ecart.idxmax()
pd.DataFrame({
    "Date": [i_max],
    "Rendement_simple": [cac_simple.loc[i_max]],
    "Log_rendement": [cac_log.loc[i_max]],
    "Ecart": [ecart.loc[i_max]],
})

L'écart maximal survient un jour de variation extrême (souvent un jour
de krach ou de forte hausse, mars 2020 dans cet échantillon) : c'est
exactement ce que prédit l'approximation $R_t \simeq \tilde R_t$ du
cours — elle n'est bonne que pour $\tilde R_t$ proche de zéro, et se
dégrade quand le rendement journalier devient grand en valeur absolue.

---

# Exercice 2 — Prix vs rendements : stationnarité visuelle

In [ ]:
#| label: prix-vs-rendements
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax[0].plot(cac_prices, color="navy", lw=1.5)
ax[0].set_title("CAC40 — Prix de clôture (non stationnaire)")
ax[0].grid(True)
ax[1].plot(cac_log, color="darkred", lw=1)
ax[1].axhline(0, ls="--", color="gray")
ax[1].set_title("CAC40 — Log-rendements journaliers (stationnaire)")
ax[1].grid(True)
plt.tight_layout(); plt.show()

Le prix affiche une tendance de long terme et des ruptures de niveau
franches (2020, 2022) : moyenne clairement non constante dans le temps.
Le rendement, lui, oscille visuellement autour d'un niveau stable proche
de zéro — cohérent avec le **fait stylisé 1** (stationnarité des
rendements, non-stationnarité des prix).

---

# Exercice 3 — Statistiques descriptives et test de normalité

## (a)-(b) Statistiques descriptives

In [ ]:
#| label: stats-descriptives
returns_vec = cac_log.dropna().to_numpy().ravel()
T_obs = len(returns_vec)

stats_df = pd.DataFrame({
    "Statistique": ["Moyenne", "Écart-type", "Asymétrie", "Kurtosis",
                    "Minimum", "Maximum"],
    "Valeur": [
        returns_vec.mean(), returns_vec.std(ddof=1),
        stats.skew(returns_vec), stats.kurtosis(returns_vec, fisher=False),
        returns_vec.min(), returns_vec.max(),
    ],
})
stats_df.round(4)

La kurtosis empirique est nettement supérieure à 3 : la distribution est
**leptokurtique**, conformément au fait stylisé 3.

## (c) Tests formels

In [ ]:
#| label: test-kurtosis-manuel
K_hat = stats.kurtosis(returns_vec, fisher=False)
stat_K = (K_hat - 3) / np.sqrt(24 / T_obs)
{"statistique": stat_K, "valeur_critique_5pc": stats.norm.ppf(0.95)}

In [ ]:
#| label: jarque-bera
jb_stat, jb_p, skew_, kurt_ = jarque_bera(returns_vec)
{"statistique": jb_stat, "p_value": jb_p}

Le test manuel sur la kurtosis et le test de Jarque-Bera rejettent tous
les deux la normalité, dans le même sens : les rendements ne sont pas
gaussiens.

## (d) Histogramme et QQ-plot

In [ ]:
#| label: histogramme-qqplot
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].hist(returns_vec, bins=50, density=True, color="lightblue")
grille = np.linspace(returns_vec.min(), returns_vec.max(), 200)
ax[0].plot(grille, stats.norm.pdf(grille, returns_vec.mean(), returns_vec.std()),
           color="red", lw=2)
ax[0].set_title("Distribution des rendements"); ax[0].set_xlabel("Rendements (%)")

stats.probplot(returns_vec, dist="norm", plot=ax[1])
ax[1].set_title("QQ-plot vs normale")
plt.tight_layout(); plt.show()

Le QQ-plot s'écarte nettement de la droite dans les deux queues :
signature typique des queues épaisses (fait stylisé 3).

---

# Exercice 4 — Absence d'autocorrélation des rendements

In [ ]:
#| label: acf-pacf-rendements
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
plot_acf(returns_vec, lags=20, ax=ax[0], title="ACF des rendements")
plot_pacf(returns_vec, lags=20, ax=ax[1], title="PACF des rendements", method="ywm")
plt.tight_layout(); plt.show()

In [ ]:
#| label: ljung-box-rendements
acorr_ljungbox(returns_vec, lags=[10])

Le test de Ljung-Box ne rejette généralement pas $H_0$ (ou marginalement) :
peu ou pas d'autocorrélation linéaire des rendements — fait stylisé 2.
Cela ne signifie pas que les rendements sont indépendants dans
le temps, comme le montre l'exercice suivant.

---

# Exercice 5 — Clustering de volatilité et effet ARCH

In [ ]:
#| label: clustering-volatilite
squared_returns = returns_vec ** 2
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax[0].plot(returns_vec, color="darkblue", lw=0.8)
ax[0].set_title("Rendements journaliers du CAC40")
ax[1].plot(squared_returns, color="darkred", lw=0.8)
ax[1].set_title("Rendements au carré (clustering de volatilité)")
plt.tight_layout(); plt.show()

In [ ]:
#| label: acf-carre-ljungbox
fig, ax = plt.subplots(figsize=(8, 4))
plot_acf(squared_returns, lags=20, ax=ax, title="ACF des rendements au carré")
plt.tight_layout(); plt.show()

acorr_ljungbox(squared_returns, lags=[10])

Contrairement aux rendements, les rendements **au carré** présentent une
autocorrélation fortement significative (rejet net de $H_0$) : c'est
l'**effet ARCH** (fait stylisé 7), modélisé explicitement au
**chapitre 3**.

---

# Exercice 6 — Asymétrie et effet de levier (aperçu)

In [ ]:
#| label: sous-periodes
dates = cac_log.index
periode_crise = (dates >= "2020-02-15") & (dates <= "2020-05-15")

{
    "ecart_type_calme": returns_vec[~periode_crise].std(ddof=1),
    "ecart_type_crise": returns_vec[periode_crise].std(ddof=1),
}

In [ ]:
#| label: correlation-levier
r_t = returns_vec[:-1]
vol_t1 = returns_vec[1:] ** 2
np.corrcoef(r_t, vol_t1)[0, 1]

L'écart-type est nettement plus élevé sur la sous-période de crise, et
la corrélation entre le rendement du jour et le carré du rendement du
lendemain est **négative** : un rendement négatif aujourd'hui tend à
être suivi d'une volatilité plus élevée demain, plus qu'un rendement
positif de même ampleur — signature de l'**effet de levier** (fait
stylisé 8), formalisée au chapitre 3 par les modèles EGARCH et
GJR-GARCH.

---

# Exercice 7 — Sur votre propre actif

In [ ]:
#| label: autre-actif
#| eval: false
# Remplacer le symbole par l'actif de votre choix, puis reprendre
# les exercices 3 à 5 à l'identique.
AUTRE = yf.download("AAPL", start="2018-01-01", end="2023-12-31", auto_adjust=True)
autre_returns = np.log(AUTRE["Close"].squeeze()).diff().dropna() * 100

> Code à réutiliser directement, avec le symbole de son choix, pour la
> partie descriptive du projet.